In [10]:
#import librosa
#import librosa.display
import numpy as np
import pandas as pd
#import ffmpeg as ff

import os
from typing import List, Tuple, Dict
#import matplotlib.pyplot as plt
import random
import cv2
import pickle
import datetime
from PIL import Image
import glob

In [11]:
data=pd.read_csv('data.csv')
datavalid=pd.read_csv('datavalid.csv')

In [12]:
def resize_image(image, new_size) -> np.ndarray:
    return cv2.resize(image, new_size, interpolation = cv2.INTER_AREA)

In [13]:
def crop_image_window(image, training) -> np.ndarray:
    height, width, _ = image.shape
    if training:
        MAX_N = height - 128
        MAX_M = width - 128
        rand_N_index, rand_M_index = random.randint(0, MAX_N) , random.randint(0, MAX_M)
        return image[rand_N_index:(rand_N_index+128),rand_M_index:(rand_M_index+128),:]
    else:
        N_index = (height - 128) // 2
        M_index = (width - 128) // 2
        return image[N_index:(N_index+128),M_index:(M_index+128),:]

In [14]:
ValueExtraversion=[]
ValueExtraversion=data['ValueExtraversion'].values
#print(ValueExtraversion)
ValueAgreeableness=[]
ValueAgreeableness=data['ValueAgreeableness'].values
#print(ValueAgreeableness)
ValueConscientiousness=[]
ValueConscientiousness=data['ValueConscientiousness'].values
#print(ValueConscientiousness)
ValueNeurotisicm=[]
ValueNeurotisicm=data['ValueNeurotisicm'].values
#print(ValueNeurotisicm)
ValueOpenness=[]
ValueOpenness=data['ValueOpenness'].values
#print(ValueOpenness)
dict1={}
z=0
w=0
videos=[]
videos=data['VideoName'].values
features=['ValueExtraversion', 'ValueAgreeableness',
       'ValueConscientiousness', 'ValueNeurotisicm', 'ValueOpenness']
features1=[ValueExtraversion, ValueAgreeableness,
       ValueConscientiousness, ValueNeurotisicm, ValueOpenness]
for x in features:
    y={}
    w=0
    for filename in videos:
        y[filename]=features1[z][w]
        w+=1
    dict1[x]=y
    z+=1

print(dict1)

{'ValueExtraversion': {'-6otZ7M-Mro.003.mp4': 0.5}, 'ValueAgreeableness': {'-6otZ7M-Mro.003.mp4': 0.6}, 'ValueConscientiousness': {'-6otZ7M-Mro.003.mp4': 0.8}, 'ValueNeurotisicm': {'-6otZ7M-Mro.003.mp4': 0.9}, 'ValueOpenness': {'-6otZ7M-Mro.003.mp4': 0.4}}


In [15]:
ValueExtraversion=[]
ValueExtraversion=datavalid['ValueExtraversion'].values
#print(ValueExtraversion)
ValueAgreeableness=[]
ValueAgreeableness=datavalid['ValueAgreeableness'].values
#print(ValueAgreeableness)
ValueConscientiousness=[]
ValueConscientiousness=datavalid['ValueConscientiousness'].values
#print(ValueConscientiousness)
ValueNeurotisicm=[]
ValueNeurotisicm=datavalid['ValueNeurotisicm'].values
#print(ValueNeurotisicm)
ValueOpenness=[]
ValueOpenness=datavalid['ValueOpenness'].values
#print(ValueOpenness)
dictvalid={}
z=0
w=0
videos=[]
videos=datavalid['VideoName'].values
features=['ValueExtraversion', 'ValueAgreeableness',
       'ValueConscientiousness', 'ValueNeurotisicm', 'ValueOpenness']
features1=[ValueExtraversion, ValueAgreeableness,
       ValueConscientiousness, ValueNeurotisicm, ValueOpenness]
for x in features:
    y={}
    w=0
    for filename in videos:
        y[filename]=features1[z][w]
        w+=1
    dictvalid[x]=y
    z+=1

print(dictvalid)

{'ValueExtraversion': {'Ymqszjv54.001.mp4': 1}, 'ValueAgreeableness': {'Ymqszjv54.001.mp4': 0.2}, 'ValueConscientiousness': {'Ymqszjv54.001.mp4': 0.5}, 'ValueNeurotisicm': {'Ymqszjv54.001.mp4': 0.6}, 'ValueOpenness': {'Ymqszjv54.001.mp4': 0.7}}


In [16]:
def reading_label_data(file_name, dictionary) -> np.ndarray:
    extracted_data = [float(dictionary[label][file_name]) for label in features]
    return np.stack(extracted_data).reshape(5,1)

In [17]:
def preprocessing_input(file_path, file_name, dict1, training):
    #Video
    image_list = []
    for filename in glob.glob(file_path):
        print(filename) #assuming gif
        im=Image.open(filename)
        np_img = np.array(im)
        print(type(np_img))
        #print(np_img.shape)
        image_list.append(np_img)
        #print((image_list))
    list1=[]
    
    resized_images = [resize_image(image= im, new_size= (248,140)) for im in image_list]
    cropped_images = [crop_image_window(resi,training) / 255.0 for resi in resized_images]
    preprocessed_video = np.stack(cropped_images)
    
    #Ground Truth
    video_gt = reading_label_data(file_name, dict1)
    del resized_images, cropped_images
    return (preprocessed_video, video_gt)

In [18]:
training_set_data = []
path = r"C:\Users\garla\Desktop\majorproject\project\videos"

for filename in data['VideoName']:
    filePath = path+'\\'+filename+"\*.jpg"
    training_set_data.append(preprocessing_input(filePath, filename,dict1,True))


C:\Users\garla\Desktop\majorproject\project\videos\-6otZ7M-Mro.003.mp4\-6otZ7M-Mro.003.mp4-00001.jpg
<class 'numpy.ndarray'>
C:\Users\garla\Desktop\majorproject\project\videos\-6otZ7M-Mro.003.mp4\-6otZ7M-Mro.003.mp4-00081.jpg
<class 'numpy.ndarray'>
C:\Users\garla\Desktop\majorproject\project\videos\-6otZ7M-Mro.003.mp4\-6otZ7M-Mro.003.mp4-00161.jpg
<class 'numpy.ndarray'>
C:\Users\garla\Desktop\majorproject\project\videos\-6otZ7M-Mro.003.mp4\-6otZ7M-Mro.003.mp4-00241.jpg
<class 'numpy.ndarray'>


In [19]:
def reshape_to_expected_input(dataset):
    
    
    x1_list = []
    y_list = []
    for i in range(0,len(dataset)):
        
        x1_list.append(dataset[i][0])
        y_list.append(dataset[i][1])
    return (np.stack(x1_list),np.stack(y_list))

In [20]:
validation_set_data = []
path = r"C:\Users\garla\Desktop\majorproject\project\videosvalid"

for filename in datavalid['VideoName']:
    filePath = path+'\\'+filename+"\*.jpg"
    validation_set_data.append(preprocessing_input(filePath,filename, dictvalid, training= False))


C:\Users\garla\Desktop\majorproject\project\videosvalid\Ymqszjv54.001.mp4\Ymqszjv54.001.mp4-00001.jpg
<class 'numpy.ndarray'>
C:\Users\garla\Desktop\majorproject\project\videosvalid\Ymqszjv54.001.mp4\Ymqszjv54.001.mp4-00081.jpg
<class 'numpy.ndarray'>
C:\Users\garla\Desktop\majorproject\project\videosvalid\Ymqszjv54.001.mp4\Ymqszjv54.001.mp4-00161.jpg
<class 'numpy.ndarray'>
C:\Users\garla\Desktop\majorproject\project\videosvalid\Ymqszjv54.001.mp4\Ymqszjv54.001.mp4-00241.jpg
<class 'numpy.ndarray'>


In [23]:
train_input = reshape_to_expected_input(training_set_data)
del training_set_data
validation_input = reshape_to_expected_input(dataset= validation_set_data)
del validation_set_data

In [25]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input, LSTM, Bidirectional, Lambda, Dropout, Concatenate
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import TimeDistributed


ModuleNotFoundError: No module named 'tensorflow'